# Faithfulness scaling — real model run (free Colab GPU)

**What this notebook does:** runs the real black-box truncation/corruption
faithfulness test from `project_overview.md` against a real
DeepSeek-R1-Distill-Qwen model, using this Colab session's own GPU instead
of a paid hosted API.

**Why this exists (instead of a hosted API key):** the daily automated
session that maintains this repo runs in a sandbox that can only reach a
short allowlist of sites (basically just PyPI and GitHub) — it cannot reach
Hugging Face, Together, Fireworks, Groq, *or* OpenRouter directly (all five
were tested and all failed the same way). That's true regardless of which
provider is picked or whether it's free, so an API key wouldn't actually
unblock that sandbox. It also turns out the OpenRouter free tier specifically
(the cheapest hosted option) only allows ~50 requests/day without ever having
spent $10 — and this experiment needs on the order of 1,000+ generation
calls per model size, so free-tier hosted APIs would take weeks per size
even if they were reachable.

Colab's free GPU has no such per-request limit (only a weekly GPU-hour cap,
which is generous relative to what one run needs), needs **no API key, no
provider signup, and no billing information** — just your existing Google
account. That's why this notebook downloads the actual model weights (they're
open-weight, no gated access needed) and runs generation locally instead.

**2026-07-20 update — segmenting across sessions/accounts:** free-tier
Colab credits can run out partway through a run (this happened on the first
real attempt, about halfway through the 1.5B model). This notebook now
checkpoints its progress *during* the run, not just at the end, and can
resume from a checkpoint from a *different* Google account. There are two
ways to move a checkpoint from one account to the next:

- **Automatic, via GitHub (recommended):** give the notebook a GitHub
  token once (section 2.5) and every checkpoint it saves is also committed
  and pushed straight to this repo. The next session, on any account,
  already has it the moment it clones the repo in section 2 — no manual
  file handling at all.
- **Manual, via upload:** without a token, checkpoints still download to
  your browser; upload the file in section 4.5 on the next account.

Either way, one model size's run can be split across as many free-tier
accounts as needed, each one picking up exactly where the last stopped.

**Before running:**
1. `Runtime` → `Change runtime type` → set **Hardware accelerator = T4 GPU**
   (free tier). If Colab offers you CPU only during peak hours, wait a bit
   and retry — this notebook will not work without a GPU.
2. `Runtime` → `Run all`.
3. If you have a partial results file from an earlier session (this
   account or another one) for the *same* model size, have it ready to
   upload when section 4.5 prompts you.
4. When it finishes, download the results file it produces (the last cell
   does this automatically) and drop it into the `results/` folder of your
   local copy of this repo, so the next automated session can pick it up,
   aggregate it, and update the scaling-curve plot.

**Which model size to run:** set `MODEL_SIZE` in the parameters cell below
and run this notebook once per size (1.5B, then 7B, then 14B) — running all
three in one Colab session in sequence works too, it just uses more of your
weekly GPU-hour budget in one sitting. All three fit on the free T4:
1.5B and 7B load in bf16 comfortably; 14B needs the 4-bit option
(`LOAD_IN_4BIT = True`, already the default for 14B below) to fit in the
T4's ~15GB usable VRAM.

## 0. Confirm you actually have a GPU runtime

If this errors or reports no GPU, go back and set
`Runtime → Change runtime type → T4 GPU` before continuing.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 1. Install dependencies

Colab already has `torch`. This adds the pieces this repo's pipeline code
needs that aren't preinstalled.

In [ ]:
%pip install -q transformers accelerate bitsandbytes datasets

## 2. Get this repo's pipeline code

Clones the public repo so this notebook can reuse the exact same
`pipeline/` code (corruption logic, scoring, prompt construction, and now
the multi-session merge/resume logic in `pipeline/merge_results.py`) that
the rest of the project uses — no duplicated logic to keep in sync.

In [ ]:
import os

REPO_URL = "https://github.com/NicolasCroft/faithfulness-scaling.git"
REPO_DIR = "faithfulness-scaling"

# GIT_TERMINAL_PROMPT=0 makes git fail fast with a clear error instead of
# hanging/erroring on a credential prompt it can't show in a non-interactive
# notebook cell -- this is what happens if the repo is private (or the URL
# is wrong): a public repo needs no credentials at all to clone.
os.environ["GIT_TERMINAL_PROMPT"] = "0"

if not os.path.exists(REPO_DIR):
    exit_code = os.system(f"git clone --depth 1 {REPO_URL}")
    if exit_code != 0 or not os.path.isdir(REPO_DIR):
        raise RuntimeError(
            "git clone failed. The most likely cause: the repo at "
            f"{REPO_URL} is private -- Colab has no GitHub credentials "
            "of yours, so cloning a private repo from here will always "
            "fail this way (no amount of retrying, zip-downloading, or "
            "changing the clone method fixes it). Fix: make the repo "
            "public on GitHub (Settings -> Danger Zone -> Change "
            "visibility), then Runtime -> Restart session and Run All "
            "again. If it's already public and this still fails, check "
            "the repo name/URL above for a typo."
        )
else:
    os.system(f"cd {REPO_DIR} && git pull")

%cd {REPO_DIR}
import sys
sys.path.insert(0, ".")

# Fail loudly here too, rather than several cells later at the `import
# pipeline` line, if the clone silently produced an unexpected layout.
assert os.path.isdir("pipeline"), (
    "Cloned the repo but no pipeline/ directory was found inside it -- "
    "something is wrong with the repo layout or REPO_DIR, not a network "
    "issue."
)
print("Repo cloned successfully; pipeline/ found.")

## 2.5 Optional: automatic checkpoint syncing via GitHub

If this cell finds a GitHub token, every checkpoint/cumulative file this
notebook saves later (section 5/6) also gets committed and pushed straight
to `results/raw/` in this repo. That means the *next* session — on this
account or a different one — already has it as soon as it clones the repo
in section 2, with no manual download/upload step at all.

Without a token this is skipped entirely and the notebook behaves exactly
as before: checkpoints still save locally and download to your browser,
and you upload them manually in section 4.5.

**One-time setup (the token is tied to your GitHub identity, not the
Google account running Colab, so you only need to do this once and can
reuse the same token in every account):**
1. On GitHub: **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Scope it to just this repo
   (`faithfulness-scaling`), permission **Contents: Read and write**, and
   give it an expiration date. A narrowly-scoped token like this is safer
   to reuse across several Colab sessions than a broad classic token.
2. In *this* Colab session (and again in every future session/account
   where you want auto-syncing): click the **key icon** in the left
   sidebar → **Secrets** → add a new secret named `GITHUB_TOKEN` → paste
   the token → toggle **Notebook access** on.
3. Run this cell (already happens automatically with Run All).

In [ ]:
import subprocess

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None

if GITHUB_TOKEN:
    # Embed the token in the origin remote so `git push` doesn't need an
    # interactive credential prompt (which a notebook cell can't answer).
    push_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.run(["git", "remote", "set-url", "origin", push_url], check=True)
    subprocess.run(["git", "config", "user.email", "colab-runner@users.noreply.github.com"], check=True)
    subprocess.run(["git", "config", "user.name", "faithfulness-scaling Colab runner"], check=True)
    print("GITHUB_TOKEN found -- checkpoints will be committed + pushed to GitHub automatically.")
else:
    print("No GITHUB_TOKEN secret found -- checkpoints will only save locally and "
          "download to your browser. Upload them manually in section 4.5 on the "
          "next account/session, or see this section's markdown to enable "
          "automatic syncing.")


def push_checkpoint(path: str) -> None:
    """Commit + push one checkpoint file to the repo, if a token was
    configured above. Never raises -- a push failure (network hiccup, stale
    remote, anything) shouldn't take down the run; the file is still safe
    locally and in the browser download either way."""
    if not GITHUB_TOKEN:
        return
    try:
        subprocess.run(["git", "add", path], check=True)
        commit = subprocess.run(
            ["git", "commit", "-m", f"checkpoint: {path}"], capture_output=True, text=True
        )
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [git commit warning] {commit.stdout.strip()} {commit.stderr.strip()}")
            return
        push = subprocess.run(["git", "push", "origin", "HEAD"], capture_output=True, text=True)
        if push.returncode == 0:
            print(f"  [pushed to GitHub] {path}")
        else:
            print(f"  [git push failed, continuing locally] {push.stderr.strip()}")
    except Exception as e:
        print(f"  [git push failed, continuing locally] {e}")

## 3. Parameters — set these, then Run All

`MODEL_SIZE` picks which of the three core sizes to run. `N_PROBLEMS` caps
how many GSM8K problems to pull; `project_overview.md` wants "at least a
few hundred correctly-solved problems per size" for the trend to not be
noise, and not every problem will be solved correctly with the full CoT, so
start higher than the target (a few hundred solved) to leave margin. 300 is
a reasonable starting point for a single Colab session; raise it if GPU-hour
budget allows and lower it if a run is taking too long or Colab disconnects
before finishing.

**If you're spreading one model size across multiple sessions/accounts**
(e.g. free-tier credits ran out partway through): keep `N_PROBLEMS` and
`SEED` the same across every session for a given model size — `N_PROBLEMS`
determines which GSM8K problems get loaded (always the first `N_PROBLEMS`
of the test split, in the same deterministic order, so `problem_id`s line
up across sessions no matter which Google account is running).
`CHECKPOINT_EVERY` controls how often (in newly-completed problems) this
notebook saves-and-downloads a partial results file *during* the run, not
just at the end — so if a session dies mid-run (credits exhausted,
disconnect, anything) you keep everything up to the last checkpoint instead
of losing the whole session. `SESSION_LABEL` is just a human-readable tag
for telling sessions apart in your Downloads folder (e.g. `"acct2"`); it's
optional and purely cosmetic.

In [ ]:
MODEL_SIZE = "1.5B"  # one of "1.5B", "7B", "14B"
N_PROBLEMS = 300
SEED = 0
CHECKPOINT_EVERY = 20  # save + download a checkpoint every N newly-completed problems
SESSION_LABEL = ""  # optional short tag (e.g. "acct2") for telling sessions apart in Downloads; purely cosmetic

MODEL_IDS = {
    "1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "7B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "14B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
}
# 14B needs 4-bit quantization to fit a free T4's ~15GB usable VRAM
# (fp16 weights alone would be ~28GB). 1.5B/7B don't need it.
LOAD_IN_4BIT = {"1.5B": False, "7B": False, "14B": True}[MODEL_SIZE]

MODEL_NAME = MODEL_IDS[MODEL_SIZE]
MODEL_KEY = MODEL_SIZE.lower()  # e.g. "1.5b" -- used in filenames and result model names
print(f"Running {MODEL_NAME} (4-bit={LOAD_IN_4BIT}) on up to {N_PROBLEMS} problems, seed={SEED}")
if SESSION_LABEL:
    print(f"Session label: {SESSION_LABEL!r}")

## 4. Load problems

`load_gsm8k` needs real internet access to download the dataset from the
Hugging Face Hub — which this Colab environment has (unlike the daily
automated sandbox), so this is expected to just work here. Loading is
deterministic (first `N_PROBLEMS` of the test split, no shuffling), which
is what makes `problem_id`s comparable across separate sessions and
accounts for the resume step below.

In [ ]:
from pipeline.data import load_gsm8k

problems = load_gsm8k(split="test", n=N_PROBLEMS)
print(f"Loaded {len(problems)} problems ({problems[0].problem_id} .. {problems[-1].problem_id})")

## 4.5 Optional: resume from a previous partial session

If a previous Colab session for **this same `MODEL_SIZE`** got cut off
partway through (free-tier credits ran out, disconnect, whatever), you
don't have to start over. This cell picks up where it left off in two
ways, and does both:

1. **Automatic:** if section 2.5's GitHub sync is set up, any checkpoint
   an earlier session (this account or a different one) already pushed is
   sitting in `results/raw/` right now, because section 2's clone pulled it
   down along with the code. This cell loads those automatically -- no
   action needed from you.
2. **Manual fallback:** for anything not already in the repo (e.g. a
   session that ran without GitHub syncing enabled), upload the
   `*_checkpoint_*.json` or `*_cumulative_*.json` file it downloaded. You
   can upload more than one at once, and files from step 1 and step 2 are
   merged together. Press Cancel/Escape on the upload dialog if step 1
   already covered everything (or if there's nothing to resume).

Either way, this cell figures out which `problem_id`s are already done and
skips them below. Works across **different Google accounts** as long as
`MODEL_SIZE`, `N_PROBLEMS`, and `SEED` match the earlier session(s).

In [ ]:
import glob
import json

from pipeline.merge_results import extract_traces, merge_trace_dicts

already_done_traces: list[dict] = []
already_done_ids: set[str] = set()
trace_lists = []


def _load_candidate(label: str, payload) -> None:
    try:
        file_traces = extract_traces(payload)
    except Exception as e:
        print(f"Skipping {label!r}: couldn't parse it as a traces file ({e}).")
        return
    # Guard against accidentally mixing in a different model size's file --
    # that would silently corrupt this run's skip-set and its final merge.
    file_model_size = payload.get("model_size") if isinstance(payload, dict) else None
    if file_model_size and file_model_size != MODEL_SIZE:
        print(f"Skipping {label!r}: it's for model_size={file_model_size!r}, not {MODEL_SIZE!r}.")
        return
    print(f"Loaded {len(file_traces)} traces from {label!r}.")
    trace_lists.append(file_traces)


# Step 1: anything already synced via GitHub (section 2.5) is already
# sitting locally after this session's clone -- no manual step needed.
for path in sorted(glob.glob(f"results/raw/{MODEL_KEY}_*.json")):
    with open(path) as f:
        _load_candidate(path, json.load(f))

# Step 2: manual fallback for anything not already in the repo.
try:
    from google.colab import files as colab_files
    print("If you have any OTHER partial-results file(s) for "
          f"MODEL_SIZE={MODEL_SIZE!r} not already listed above, upload them "
          "now -- otherwise cancel/press Escape.")
    uploaded = colab_files.upload()
except ImportError:
    uploaded = {}
    print("Not running in Colab -- skipping the manual upload step.")

for fname, raw_bytes in uploaded.items():
    try:
        payload = json.loads(raw_bytes.decode("utf-8"))
    except Exception as e:
        print(f"Skipping {fname!r}: not valid JSON ({e}).")
        continue
    _load_candidate(fname, payload)

if trace_lists:
    already_done_traces, resume_conflicts = merge_trace_dicts(trace_lists)
    already_done_ids = {t["problem_id"] for t in already_done_traces}
    if resume_conflicts:
        print(f"WARNING: {len(resume_conflicts)} problem_id(s) disagreed across uploaded files:")
        for c in resume_conflicts:
            print(f"  {c}")
    print(f"Resuming: {len(already_done_ids)} problems already done, will be skipped below.")
else:
    print("No usable resume files -- starting this model size from scratch.")

remaining_problems = [p for p in problems if p.problem_id not in already_done_ids]
print(f"{len(remaining_problems)}/{len(problems)} problems remain to process in this session.")

## 4.6 Load the model

Downloads and loads the actual model weights for `MODEL_SIZE`. This is the
slow, one-time-per-session setup step.

In [ ]:
from pipeline.inference import LocalHFBackend

backend = LocalHFBackend(
    model_name=MODEL_NAME,
    load_in_4bit=LOAD_IN_4BIT,
)
# Trigger the actual weight download + load now (rather than lazily on the
# first generate() call) so any failure surfaces here, before the long
# run, not partway through it.
backend._load()
print("Model loaded.")

## 5. Run the truncation/corruption faithfulness test

This is the real experiment: generate CoT + answer for each *remaining*
problem (already-done ones from an uploaded resume file are skipped), keep
only the ones solved correctly, apply the standard corruption battery, and
regenerate. This is the slow cell — expect it to take a while (roughly
proportional to model size x number of problems).

Two things changed from a plain "run everything then save once at the end"
loop, both aimed at surviving free-tier credit limits without losing work:

1. **Per-problem error handling** — if one problem throws (a weird
   generation edge case, a transient OOM, etc.) it's recorded as a failed
   trace and the loop moves on, instead of the whole run dying on one bad
   problem.
2. **Periodic checkpointing** — every `CHECKPOINT_EVERY` newly-completed
   problems, this cell saves everything done so far (earlier sessions'
   traces plus this session's), downloads it to your browser, and -- if
   section 2.5's GitHub sync is set up -- pushes it to the repo too. If
   the session gets cut off before reaching the final save cell in
   section 6, that work isn't lost: either the next session's clone
   already has it (GitHub path), or you upload the file from your
   Downloads folder in section 4.5 (manual path).

Progress prints every 25 problems; checkpoint saves print their own line so
you can see exactly which file was just written.

In [ ]:
import os
import time
from datetime import datetime, timezone

from pipeline.run_experiment import run_for_problem

os.makedirs("results/raw", exist_ok=True)


def _trace_to_dict(t) -> dict:
    return {
        "problem_id": t.problem.problem_id,
        "question": t.problem.question,
        "gold_answer": t.problem.answer,
        "model_answer": t.original.final_answer,
        "solved_correctly": t.solved_correctly,
        "corruption_outcomes": [
            {"method": c.method, "new_answer": a, "answer_changed": ch}
            for c, a, ch in t.corruption_outcomes
        ],
    }


def _failed_trace_dict(problem, error) -> dict:
    return {
        "problem_id": problem.problem_id,
        "question": problem.question,
        "gold_answer": problem.answer,
        "model_answer": None,
        "solved_correctly": False,
        "corruption_outcomes": [],
        "error": str(error),
    }


def save_checkpoint(all_traces_so_far: list[dict], tag: str) -> str:
    """Write + (if in Colab) download a self-contained snapshot of every
    trace gathered so far -- both from earlier sessions (already_done_traces)
    and this session (session_new_traces). Self-contained on purpose: this
    one file is enough to resume from next time, no need to keep every
    older checkpoint around too."""
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    label_part = f"_{SESSION_LABEL}" if SESSION_LABEL else ""
    path = f"results/raw/{MODEL_KEY}{label_part}_{tag}_{timestamp}.json"
    payload = {
        "model_size": MODEL_SIZE,
        "model_name": MODEL_NAME,
        "load_in_4bit": LOAD_IN_4BIT,
        "seed": SEED,
        "n_problems_target": N_PROBLEMS,
        "n_problems_in_file": len(all_traces_so_far),
        "traces": all_traces_so_far,
    }
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    try:
        from google.colab import files as colab_files
        colab_files.download(path)
        print(f"  [saved + downloaded] {path} ({len(all_traces_so_far)} traces total)")
    except ImportError:
        print(f"  [saved locally] {path} ({len(all_traces_so_far)} traces total) -- not in Colab, download manually")
    push_checkpoint(path)  # no-op if section 2.5 didn't find a GITHUB_TOKEN
    return path


start = time.time()
session_new_traces: list[dict] = []
since_last_checkpoint = 0

for i, p in enumerate(remaining_problems):
    try:
        trace = run_for_problem(p, backend, seed=SEED)
        session_new_traces.append(_trace_to_dict(trace))
    except Exception as e:
        print(f"  [error on {p.problem_id}] {e} -- recording as failed and continuing.")
        session_new_traces.append(_failed_trace_dict(p, e))
    since_last_checkpoint += 1

    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (len(remaining_problems) - (i + 1)) / rate if rate > 0 else float("nan")
        print(f"{i + 1}/{len(remaining_problems)} new problems done this session "
              f"({len(already_done_ids) + i + 1}/{len(problems)} cumulative), "
              f"{elapsed/60:.1f} min elapsed, ~{remaining/60:.1f} min remaining")

    if since_last_checkpoint >= CHECKPOINT_EVERY:
        save_checkpoint(already_done_traces + session_new_traces, tag="checkpoint")
        since_last_checkpoint = 0

solved_this_session = sum(1 for t in session_new_traces if t["solved_correctly"])
print(f"\nThis session: solved correctly with full CoT: {solved_this_session}/{len(session_new_traces)}")

## 6. Save results

Merges what was already done in earlier sessions (uploaded in 4.5, if any)
with what this session just produced, recomputes the aggregate faithfulness
summary from that full union (via `pipeline.merge_results`, so the math is
identical to what one uninterrupted session would have produced), and
saves + downloads **one cumulative file** for this model size.

**Sanity-check note:** if the cumulative solved-count is well under "a few
hundred," the summary numbers are noisier than `project_overview.md` wants
to report as a finding — that's a judgment call for a human to make when
looking at the actual count, not something to silently accept. The progress
report printed below tells you where you stand against the `N_PROBLEMS`
target.

In [ ]:
from pipeline.merge_results import aggregate, progress_report
from pipeline.scoring import summarize

all_traces = already_done_traces + session_new_traces
model_name_for_results = f"deepseek-r1-distill-qwen-{MODEL_KEY}"
results = aggregate(all_traces, model_name=model_name_for_results)

cumulative_path = save_checkpoint(all_traces, tag="cumulative")

print()
print(summarize(results))
print()
print(progress_report(all_traces, target_n_problems=N_PROBLEMS))

## 7. Next step

**If you set up GitHub syncing in section 2.5:** nothing to do here --
every checkpoint and the final cumulative file already got pushed to
`results/raw/` as it went. Skip to the multi-account steps below if
`N_PROBLEMS` isn't fully covered yet; otherwise you're done with this
model size.

**If you didn't set up GitHub syncing:** move the cumulative
`*_cumulative_*.json` file this cell just downloaded into the
`results/raw/` folder of your local copy of this repo (and commit/push it
yourself, if you want the next session to see it via `git pull` too). The
next automated session (or you, manually) can then read it, aggregate
across model sizes, and regenerate the scaling-curve plot.

**If `N_PROBLEMS` isn't fully covered yet** (check the progress report
printed in section 6) — e.g. credits ran out and only part of this model
size got done — here's the multi-account workflow:

1. If GitHub syncing is set up, just open this notebook in the next
   account with the same `MODEL_SIZE`/`N_PROBLEMS`/`SEED` and
   `Runtime → Run all` -- section 2's clone and section 4.5's auto-scan
   pick up everything already pushed, automatically.
2. Without GitHub syncing: keep whichever file this session most recently
   downloaded (a `*_checkpoint_*` if the run got cut off mid-loop, or the
   `*_cumulative_*` from section 6 if the session finished normally but the
   progress report still shows fewer than `N_PROBLEMS`), open the notebook
   in the next account, and upload that file in section 4.5 when prompted.
3. `Runtime → Run all`. The resume step skips every `problem_id` already
   covered, so this session only does the remaining work — spread across
   however many accounts that takes.
4. Once the progress report in section 6 shows the full `N_PROBLEMS` (or
   at least clears "a few hundred" solved correctly), that model size is
   done.

Repeat this whole notebook (sections 3–7) once per model size — `1.5B`,
then `7B`, then `14B` — each with its own chain of sessions/accounts as
needed.